# Langextract aplication with atributes

In [1]:
import os
import re
import timeit
import random
import textwrap
import unicodedata
import pandas as pd
import langextract as lx
from rich.pretty import pprint
from dotenv import load_dotenv
from more_itertools import unique_justseen
from aymurai.database.utils import text_to_uuid
from aymurai.api.endpoints.routers.misc.document_extract import extraction

In [2]:
import logfire

logfire.configure(service_name="aymurai")
_ = logfire.instrument_openai()

Logfire project URL: ]8;id=599152;https://logfire-us.pydantic.dev/sofia-delpozo/aymurai\https://logfire-us.pydantic.dev/sofia-delpozo/aymurai]8;;\


In [3]:
VALID_CLASSES = {
    "BANCO","CBU","CORREO_ELECTRONICO","CUIT_CUIL","CUIJ","DIRECCION","DNI","EDAD",
    "ESTUDIOS","FECHA","LINK","LOC","MARCA_AUTOMOVIL","NACIONALIDAD","NUM_CAJA_AHORRO",
    "NUM_EXPEDIENTE","NUM_MATRICULA","PATENTE_DOMINIO","PER","NUM_ACTUACION","TELEFONO","RELACION"
}
PERSON_ATTRS = {"DNI","CUIT_CUIL","DIRECCION","TELEFONO","CORREO_ELECTRONICO","EDAD","NACIONALIDAD","NUM_MATRICULA"}
MAIN_DF ='documents0203entrerios-docs02-08-sin05.csv' #'df-NER-vals-02-08-sin05.csv'
DOCS_PATH = '/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' ##'/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/'#'/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' #

In [4]:
df = pd.read_csv(MAIN_DF)
df.head()

,text,prediction,validation,id_x,created_at_x,updated_at,id_y,document_id,paragraph_id,order,created_at_y,name
0,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,57f85770879c420d8de2a0f3267b653b,518a7f34ad865b91ad95d954093f091a,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-22 21:20:53-03:00,document-02.docx
1,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,768db7897dc943489e9570a117975fe1,2536b5d2111d55c6b545e6bcbb75e002,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 01:00:28-03:00,document-08.docx
2,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,2c74793eac2341969aa3dcce7950f43b,6a7d2422da6a5852b68a7bee678d1aae,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 02:56:14-03:00,document-04.docx
3,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,b31e11b3cff24069b7cf3faa666fbdec,0aec75365ad0511698f72bacb8b88212,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 19:41:55-03:00,document-03.docx
4,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,4abeb6f8ff3340cb99cfe61b16276012,518a7f34ad865b91ad95d954093f091a,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 21:20:53-03:00,document-02.docx


In [5]:
set(df.name)

{'aymurai - ejemplo 02.docx',
 'aymurai - ejemplo 03.docx',
 'document-02.docx',
 'document-03.docx',
 'document-04.docx',
 'document-06.docx',
 'document-07.docx',
 'document-08.docx',
 nan}

## Prompt & example definitions

In [89]:
PROMPT_V8 = textwrap.dedent("""Sos un extractor y clasificador de ENTIDADES en documentos judiciales en español.
Leé TODO el texto. Devolvé SOLO spans EXACTOS (sin parafrasear ni inferir).

DEVOLUCIÓN **ÚNICA**:
RESPONDÉ ÚNICAMENTE con un JSON válido con la forma EXACTA:
{"extractions":[
  {"extraction_class": "...", "extraction_text": "...", "group_index": 0, "attributes": { ...? }},
  ...
]}
- No incluyas texto extra, ni comentarios, ni listas, ni bloques “attributes: …” como líneas aparte.
- "attributes" es siempre un DICCIONARIO *dentro* del mismo item. NUNCA devuelvas una extracción con clases "attributes", "attrs", "group_index", "extraction_text" u otras no definidas.
- Si no hay extracciones: {"extractions": []}

CLASES VÁLIDAS (set cerrado):
BANCO, CBU, CORREO_ELECTRONICO, CUIT_CUIL, CUIJ, DIRECCION, DNI, EDAD, ESTUDIOS, FECHA,
LINK, LOC, MARCA_AUTOMOVIL, NACIONALIDAD, NUM_CAJA_AHORRO, NUM_EXPEDIENTE, NUM_MATRICULA,
PATENTE_DOMINIO, PER, NUM_ACTUACION, TELEFONO, RELACION

REGLAS DE EXTRACCIÓN (resumen):
- NUM_EXPEDIENTE: extraé \d+/\d{4} SOLO si está junto a “Expediente/Expte./Exp./Exp. N°/Nº/N.o”.
- DNI: 7–8 dígitos (#######, ######## o ##.###.###). No extraer la palabra “DNI” sola.
- DIRECCION: calle+altura (“9 de Julio 456”), intersección (“Callao y Corrientes”) o número claro de domicilio.
- FECHA: dd/mm/aaaa, dd-mm-aaaa, “dd de <mes> de aaaa”, compuestas (“5 y 7 de mayo de 2020”), “tercer día de marzo”.
- LOC: localidad / provincia / país / continente (no hospitales ni domicilios).
- PER: nombre completo, iniciales o apodo inequívoco. **También dentro de comillas**: tratá el texto entre comillas como texto normal y extraé PER si aparecen (p. ej., carátulas o citas).
- RELACION (medida/vínculo explícito). "attributes" puede incluir:
  - "TIPO" ∈ {"prohibicion_acercamiento","cese_perturbacion","contacto_prohibido","orden_cautelar","restriccion_perimetral"}
  - "SUJETO_ACTIVO_GROUP": <int>, "SUJETO_PASIVO_GROUP": <int>, y opcionales como "DISTANCIA_MIN_M": <int> si aparece.

ATRIBUTOS de PER (en "attributes" del MISMO item):
- "ROL_PROCESAL" si es inequívoco: {"Denunciante","Imputado","Victima","Juez","Fiscal","Secretario","Prosecretario","Mediador",
  "Asesor Tutelar","Damnificado","Querellante","Actor","Demandado","Testigo","Perito","Defensor Oficial","Defensor Particular",
  "Apoderado","Tutor","Curador","Policia","Perjudicado","Beneficiario"}
- "TIPO" si corresponde (p.ej., "conyugue_de_la_victima","hijo/a_de_la_victima","hija_menor").

AGRUPAMIENTO (group_index):
- Identifica PERSONAS (PER). Política determinista:
  * La PRIMERA vez que aparece una nueva persona recibe el menor entero libre (0,1,2,...).
  * TODAS sus repeticiones reutilizan SIEMPRE el MISMO group_index.
- ATRIBUTOS personales pegados a una PER en la MISMA oración/cláusula heredan el group_index de esa PER:
  DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- “Sticky”: patrones que obligan herencia:
  "<PER>, DNI <número>", "<PER>, con domicilio en <span>", "la Sra./el Sr. <PER> ... DNI/domc." en la misma oración.
- Considerá igual persona aunque varíe el orden (“Morales, Julieta Andrea” ≡ “Julieta Andrea Morales”) o falten tildes.

CALIDAD Y COHERENCIA:
- No superpongas spans. No inventes datos ni normalices.
- No extraigas clases no listadas. **Nunca** devuelvas ítems con "extraction_class": "attributes"/"group_index"/"extraction_text".
- Al finalizar: si la misma persona recibió más de un group_index, UNIFICAR al primero y corregir en todos los ítems del JSON.
""")

In [84]:
PROMPT_V7 = textwrap.dedent("""Sos un extractor y clasificador de ENTIDADES en documentos judiciales en español.
Leé TODO el texto. Devolvé SOLO spans EXACTOS sin parafrasear ni inferir.

FORMATO DE SALIDA (obligatorio):
- Devolvé una lista JSON bajo la clave "extractions": {"extractions":[ ... ]}.
- Cada item es: {"extraction_class": <CLASE>, "extraction_text": <SPAN>, "group_index": <int>, "attributes": { ...? }}.
- "group_index" es un entero (0,1,2,...) usado SOLO para AGRUPAR referencias a la MISMA PERSONA (PER) y SUS ATRIBUTOS.
- "attributes" es un diccionario opcional SOLO dentro del item (NO crear entradas separadas).
- PROHIBIDO crear items con clases NO listadas abajo. EN ESPECIAL: NUNCA crear items con clases "group_index", "attrs", "attributes", "role" u otras inventadas.

CLASES VÁLIDAS (set cerrado):
BANCO, CBU, CORREO_ELECTRONICO, CUIT_CUIL, CUIJ, DIRECCION, DNI, EDAD, ESTUDIOS, FECHA,
LINK, LOC, MARCA_AUTOMOVIL, NACIONALIDAD, NUM_CAJA_AHORRO, NUM_EXPEDIENTE, NUM_MATRICULA,
PATENTE_DOMINIO, PER, NUM_ACTUACION, TELEFONO, RELACION

REGLAS DE EXTRACCIÓN (resumen):
- NUM_EXPEDIENTE: extraé \d+/\d{4} SOLO si aparece junto a “Expediente/Expte./Exp./Exp. N°/Nº/N.o”.
- DNI: 7–8 dígitos en formatos #######, ######## o ##.###.### (no extraer la palabra "DNI" sola).
- DIRECCION: calle+altura (“9 de Julio 456”), intersección (“Callao y Corrientes”) o número suelto cuando claramente es domicilio.
- FECHA: dd/mm/aaaa, dd-mm-aaaa, “dd de <mes> de aaaa”, compuestas (“5 y 7 de mayo de 2020”), expresiones como “tercer día de marzo”.
- LOC: localidad/provincia/país/continente (no hospitales ni domicilios).
- PER: nombre completo, iniciales o apodo inequívoco.
- RELACION: solo si hay una medida/vínculo explícito; "attributes" puede incluir:
  - "TIPO" ∈ {"prohibicion_acercamiento","cese_perturbacion","contacto_prohibido","orden_cautelar","restriccion_perimetral"}
  - "SUJETO_ACTIVO_GROUP": <int>, "SUJETO_PASIVO_GROUP": <int>, y opcionales como "DISTANCIA_MIN_M": <int> si aparece.

ATRIBUTOS PER (dentro de "attributes"):
- "ROL_PROCESAL" si es inequívoco: {"Denunciante","Imputado","Victima","Juez","Fiscal","Secretario","Prosecretario","Mediador",
  "Asesor Tutelar","Damnificado","Querellante","Actor","Demandado","Testigo","Perito","Defensor Oficial","Defensor Particular",
  "Apoderado","Tutor","Curador","Policia","Perjudicado","Beneficiario"}
- "TIPO" si es claro (p. ej., "conyugue_de_la_victima","hijo/a_de_la_victima","hija_menor", etc.)

AGRUPAMIENTO (group_index):
- El group_index IDENTIFICA PERSONAS (PER). Asignación determinista:
  * La PRIMERA aparición de una nueva persona recibe el menor entero no usado (0,1,2,...).
  * TODAS las repeticiones de esa persona reutilizan SIEMPRE el MISMO group_index.
- Los ATRIBUTOS personales pegados a una PER (en la MISMA oración o cláusula) heredan el group_index de ESA PER:
  DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- "Sticky": si el patrón es “<PER>, DNI <número> ...”, “<PER>, con domicilio en <span> ...” o
  “la Sra./el Sr. <PER> ... DNI/ domicilio ...” en la MISMA oración → el atributo usa el group_index de esa PER.
- Considerá igual persona aunque varíe el orden (“Morales, Julieta Andrea” ≡ “Julieta Andrea Morales”) o haya tildes.

REGLAS DE CALIDAD:
- No superpongas spans. No inventes datos ni normalices.
- No extraigas “DNI” sin número ni clases no listadas.
- Si NO hay nada, devolvé {"extractions": []} (lista vacía).
- Al finalizar, verificá coherencia: una misma persona NO puede recibir dos group_index distintos; si detectás conflicto, unificá al primero asignado y corregí todos los items.

Ejemplos se proporcionan aparte. """)


In [ ]:
MERGED_PROMPTv6 = textwrap.dedent("""
Sos un extractor y clasificador de ENTIDADES en documentos judiciales en español.
Leé el documento completo, identificá las entidades que mencionaremos a continuacion, devolvé SOLO spans EXACTOS (sin parafrasear ni inferir) de los textos,
asignandole su clase, sus atributos (si corresponde) y el grupo al que pertenecen. Identifica las personas a traves del documento, para asociarlas a un mismo grupo si aparecen más de una vez.
                                  
Si una clase NO aparece, no devuelvas nada. No superpongas entidades (una mención = una extracción).
                                  
LAS UNICAS CLASES VÁLIDAS SON:
- BANCO: entidad bancaria (no vale solo “Banco”).
- CBU: número de 22 dígitos.
- CORREO_ELECTRONICO
- CUIT_CUIL: ##-########-#.
- CUIJ: código judicial ##-########-#.
- DIRECCION: calle+altura (“9 de Julio 456”), intersección (“Callao y Corrientes”), o número suelto.
- DNI: 7-8 dígitos, #######, ########, ##.###.### (no solo la palabra “DNI”).
- EDAD: edad en años o meses.
- ESTUDIOS: nivel educativo (primario, secundario, terciario, universitario, posgrado, doctorado; puede incluir “incompleto”, “completo”, “finalizado”, “en curso”).
- FECHA: en multiples formatos como dd/mm/aaaa, dd-mm-aaaa, “dd de <mes> de aaaa”, compuestas (ej. “5 y 7 de mayo de 2020”), o por ej. "tercer día de marzo"  (no expresiones vagas como “ayer” o “a las 15:00 horas”).
- LINK: URL.
- LOC: localidad, provincia, país, continente (no hospitales, domicilios ni referencias genéricas).
- MARCA_AUTOMOVIL
- NACIONALIDAD
- NUM_CAJA_AHORRO
- NUM_EXPEDIENTE: extraé \\d+/\\d{4} cuando aparezca junto a “Expediente”, “Expte.”, “Exp.”, “Exp. N°/Nº/N.o”, o análogo.
- NUM_MATRICULA
- PATENTE_DOMINIO: dominio de vehículo (formatos AAA123 o AA123AA).
- PER: nombre completo, iniciales o apodo de persona física.
- NUM_ACTUACION 
- TELEFONO
- RELACION: vínculo o medida entre personas

ATRIBUTOS de la clase PER:
  - ROL_PROCESAL solo si esta claro en el texto:
    • “denunciante/denunció” ⇒ {"ROL_PROCESAL":"Denunciante"}
    • “denunciado”, “imputado”, “acusado” ⇒ {"ROL_PROCESAL":"Imputado"}
    • “víctima” ⇒ {"ROL_PROCESAL":"Victima"}
    • “jueza/juez” ⇒ {"ROL_PROCESAL":"Juez"}
    • Otros posibles: "Fiscal","Secretario","Prosecretario","Mediador","Asesor Tutelar","Damnificado","Querellante",
      "Actor","Demandado","Testigo","Perito","Defensor Oficial","Defensor Particular","Apoderado","Tutor","Curador",
      "Policia","Perjudicado","Beneficiario".
  - TIPO solo si esta claro en el texto (ej.: "conyugue_de_la_victima","hijo/a_de_la_victima").
ATRIBUTOS de la clase DIRECCION:
 - TIPO, solo si esta claro en el texto, valores: "domicilio_real","domicilio_legal","domicilio_laboral",
  "domicilio_de_la_victima","domicilio_del_imputado","domicilio_del_testigo".
                                  
AGRUPAMIENTO
- La misma persona (PER) y sus ATRIBUTOS comparten el mismo indice: DNI, CUIT_CUIL, DIRECCION, TELEFONO,
  CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- Si aparece una PER sin atributos, igual asignale un indice y usalo en TODAS sus repeticiones.
- Considerá la misma persona aunque cambie el orden (p.ej., “Morales, Julieta Andrea” ≡ “Julieta Andrea Morales”).

REGLA DE AGRUPAMIENTO “STICKY”
- Si un ATRIBUTO personal (DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA)
  aparece a continuación de una PER en la MISMA oración o cláusula (separado por comas o por frases como “DNI”, “con domicilio en”),
  debe usar el MISMO indice de grupo de esa PER.
- No crees un indice de grupo nuevo para un ATRIBUTO si existe una PER inmediatamente precedente en la misma oración.
- Patrones que obligan a “pegar” el atributo a la última PER:
  • "<PER>, DNI <número>" → DNI usa el indice de grupo de esa PER.
  • "<PER>, con domicilio en <span>" → DIRECCION usa el indice de grupo de esa PER.
  • "la Sra./el Sr. <PER> ..." seguido de "DNI ..." o "domicilio ..." en la misma oración → pegar al indice de grupo de esa PER.

REGLAS GENERALES
- Aceptá mayúsculas, tildes faltantes, comillas y formatos no estándar si el span es claro.
- Carátulas/encabezados: en “Apellido, Nombre c/ Apellido, Nombre s/ …” extraé ambos PER con comas y orden exacto,
  aunque estén entre comillas. Si en la misma frase aparece “Expte./Expediente …”, extraé también el NUM_EXPEDIENTE.
- Edades: aceptá “9 años”, “9 años de edad”, “de 9 años” (extraé el núcleo “9 años”) y vinculala al indice de grupo de la persona.
- NUM_EXPEDIENTE: extraé \\d+/\\d{4} cuando aparezca junto a “Expediente”, “Expte.”, “Exp.”, “Exp. N°/Nº/N.o”.
- Si una entidad está mal escrita pero es inequívoca (ej.: en comillas, en carátula), extraela igual.
- No inventes índices para atributos: los atributos personales DEBEN reutilizar el indice de grupo de la última PER mencionada
  en la misma oración/cláusula, salvo que el texto asocie explícitamente ese atributo a otra persona.

RELACION (medidas/vínculos)
- class = RELACION solo si hay una medida o vínculo explícito.
- text = fragmento EXACTO de la medida.
- attrs posibles:
  - "TIPO" ∈ {"prohibicion_acercamiento","cese_perturbacion","contacto_prohibido","orden_cautelar","restriccion_perimetral"}
  - "SUJETO_ACTIVO_GROUP" (int), "SUJETO_PASIVO_GROUP" (int)

Luego de asignarle a cada elemento su clase, su indice de grupo y si aplica atributo, chequear que la misma persona este siempre en el mismo indice de grupo, y tenga el mismo atributo, sino corregir. 

Evitar ERRORES como:
  Extraction(extraction_class='PER', extraction_text='Rodríguez, Ana',  group_index=1, description=None)
  Extraction(extraction_class='PER', extraction_text='Ana Rodríguez',  group_index=2, description=None, attributes={'ROL_PROCESAL': 'Victima'})
  Donde claramente son la misma persona y le asigna grupos y atributos diferentes.

  De un texto como: "Clara Martinez DNI 25432123 fue victima de violencia. Se deja en acta el dia de 12/12/2010 que..."                                                                
  Extraction(extraction_class='PER', extraction_text='Clara Martinez',  group_index=1, description=None, attributes={'ROL_PROCESAL': 'Victima'})
  Extraction(extraction_class='FECHA', extraction_text='12/12/2010',  group_index=2)
  Extraction(extraction_class='DNI', extraction_text='25432123',  group_index=2)
  Donde claramente el DNI tiene que tener el grupo de la persona y no de la fecha.
                                                                
EJEMPLOS
1) Carátula: "Garcia, Maria  c/ Gutierrez, Jose s/ Violencia Familiar"
   → PER("Garcia, Maria", group_index=0)
   → PER("Gutierrez, Jose", group_index=1)

2) Roles: “la Sra. Maria Garcia denunció a Jose Gutierrez”
   → PER("Maria Garcia", group_index=0, attrs={"ROL_PROCESAL":"Denunciante"})
   → PER("Jose Gutierrez", group_index=1, attrs={"ROL_PROCESAL":"Imputado"})

3) Atributos: Sara  Morales, DNI 35.678.912, con domicilio en 9 de Julio 456…”
   → PER("Sara Morales", group_index=0)
   → DNI("35.678.912", group_index=0)
   → DIRECCION("9 de Julio 456", group_index=0)

4) Medida: “Prohibición de acercamiento del denunciado a la denunciante en un radio de 500 metros.”
   → RELACION("Prohibición de acercamiento del denunciado a la denunciante en un radio de 500 metros",
              attrs={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0,"DISTANCIA_MIN_M":500})

5) Sticky en la misma oración: “la Sra. Ana Lopez, DNI 39112456, con domicilio en calle Belgrano 785…”
   → PER("Ana  Lopez", group_index=0)
   → DNI("39112456", group_index=0)
   → DIRECCION("calle Belgrano 785", group_index=0)

6) Carátula entre comillas + Expte: “… en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.”
   → PER("Rodríguez, Ana Carolina", group_index=0)
   → PER("Fernández, Diego Esteban", group_index=1)
   → NUM_EXPEDIENTE("3187/2023")
""")

In [74]:
MERGED_PROMPTv5 = textwrap.dedent("""
Sos un extractor y clasificador de ENTIDADES en documentos judiciales en español.
Leé el documento completo, identificá las entidades que mencionaremos a continuacion, devolvé SOLO spans EXACTOS (sin parafrasear ni inferir) de los textos,
asignandole su clase, sus atributos (si corresponde) y el grupo al que pertenecen.
Si una clase NO aparece, no devuelvas nada. No superpongas entidades (una mención = una extracción).
                                  
LAS UNICAS CLASES VÁLIDAS SON:
- BANCO: entidad bancaria (no vale solo “Banco”).
- CBU: número de 22 dígitos.
- CORREO_ELECTRONICO
- CUIT_CUIL: ##-########-#.
- CUIJ: código judicial ##-########-#.
- DIRECCION: calle+altura (“9 de Julio 456”), intersección (“Callao y Corrientes”), o número suelto.
- DNI: 7-8 dígitos, #######, ########, ##.###.### (no solo la palabra “DNI”).
- EDAD: edad en años o meses.
- ESTUDIOS: nivel educativo (primario, secundario, terciario, universitario, posgrado, doctorado; puede incluir “incompleto”, “completo”, “finalizado”, “en curso”).
- FECHA: en multiples formatos como dd/mm/aaaa, dd-mm-aaaa, “dd de <mes> de aaaa”, compuestas (ej. “5 y 7 de mayo de 2020”), o por ej. "tercer día de marzo"  (no expresiones 
  vagas como “ayer” o “a las 15:00 horas”).
- LINK: URL.
- LOC: localidad, provincia, país, continente (no hospitales, domicilios ni referencias genéricas).
- MARCA_AUTOMOVIL
- NACIONALIDAD
- NUM_CAJA_AHORRO
- NUM_EXPEDIENTE: extraé \\d+/\\d{4} cuando aparezca junto a “Expediente”, “Expte.”, “Exp.”, “Exp. N°/Nº/N.o”, o análogo.
- NUM_MATRICULA
- PATENTE_DOMINIO: dominio de vehículo (formatos AAA123 o AA123AA).
- PER: nombre completo, iniciales o apodo de persona física.
- NUM_ACTUACION 
- TELEFONO
- RELACION: vínculo o medida entre personas

ATRIBUTOS de la clase PER:
  - ROL_PROCESAL solo si esta claro en el texto:
    • “denunciante/denunció” ⇒ {"ROL_PROCESAL":"Denunciante"}
    • “denunciado”, “imputado”, “acusado” ⇒ {"ROL_PROCESAL":"Imputado"}
    • “víctima” ⇒ {"ROL_PROCESAL":"Victima"}
    • “jueza/juez” ⇒ {"ROL_PROCESAL":"Juez"}
    • Otros posibles: "Fiscal","Secretario","Prosecretario","Mediador","Asesor Tutelar","Damnificado","Querellante",
      "Actor","Demandado","Testigo","Perito","Defensor Oficial","Defensor Particular","Apoderado","Tutor","Curador",
      "Policia","Perjudicado","Beneficiario".
  - TIPO solo si esta claro en el texto (ej.: "conyugue_de_la_victima","hijo/a_de_la_victima").
ATRIBUTOS de la clase DIRECCION:
 - TIPO, solo si esta claro en el texto, valores: "domicilio_real","domicilio_legal","domicilio_laboral",
  "domicilio_de_la_victima","domicilio_del_imputado","domicilio_del_testigo".
                                  
AGRUPAMIENTO
- La misma persona (PER) y sus ATRIBUTOS comparten el mismo group_index: DNI, CUIT_CUIL, DIRECCION, TELEFONO,
  CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- Si aparece una PER sin atributos, igual asignale un group_index y usalo en TODAS sus repeticiones.
- Considerá la misma persona aunque cambie el orden (p.ej., “Morales, Julieta Andrea” ≡ “Julieta Andrea Morales”).

REGLA DE AGRUPAMIENTO “STICKY”
- Si un ATRIBUTO personal (DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA)
  aparece a continuación de una PER en la MISMA oración o cláusula (separado por comas o por frases como “DNI”, “con domicilio en”),
  debe usar el MISMO group_index de esa PER.
- No crees un group_index nuevo para un ATRIBUTO si existe una PER inmediatamente precedente en la misma oración.
- Patrones que obligan a “pegar” el atributo a la última PER:
  • "<PER>, DNI <número>" → DNI usa el group_index de esa PER.
  • "<PER>, con domicilio en <span>" → DIRECCION usa el group_index de esa PER.
  • "la Sra./el Sr. <PER> ..." seguido de "DNI ..." o "domicilio ..." en la misma oración → pegar al group_index de esa PER.

REGLAS GENERALES
- Aceptá mayúsculas, tildes faltantes, comillas y formatos no estándar si el span es claro.
- Carátulas/encabezados: en “Apellido, Nombre c/ Apellido, Nombre s/ …” extraé ambos PER con comas y orden exacto,
  aunque estén entre comillas. Si en la misma frase aparece “Expte./Expediente …”, extraé también el NUM_EXPEDIENTE.
- Edades: aceptá “9 años”, “9 años de edad”, “de 9 años” (extraé el núcleo “9 años”) y vinculala al group_index de la persona.
- NUM_EXPEDIENTE: extraé \\d+/\\d{4} cuando aparezca junto a “Expediente”, “Expte.”, “Exp.”, “Exp. N°/Nº/N.o”.
- Si una entidad está mal escrita pero es inequívoca (ej.: en comillas, en carátula), extraela igual.
- No inventes índices para atributos: los atributos personales DEBEN reutilizar el group_index de la última PER mencionada
  en la misma oración/cláusula, salvo que el texto asocie explícitamente ese atributo a otra persona.

RELACION (medidas/vínculos)
- class = RELACION solo si hay una medida o vínculo explícito.
- text = fragmento EXACTO de la medida.
- attrs posibles:
  - "TIPO" ∈ {"prohibicion_acercamiento","cese_perturbacion","contacto_prohibido","orden_cautelar","restriccion_perimetral"}
  - "SUJETO_ACTIVO_GROUP" (int), "SUJETO_PASIVO_GROUP" (int)

EJEMPLOS
1) Carátula: "Garcia, Maria  c/ Gutierrez, Jose s/ Violencia Familiar"
   → PER("Garcia, Maria", group_index=0)
   → PER("Gutierrez, Jose", group_index=1)

2) Roles: “la Sra. Maria Garcia denunció a Jose Gutierrez”
   → PER("Maria Garcia", group_index=0, attrs={"ROL_PROCESAL":"Denunciante"})
   → PER("Jose Gutierrez", group_index=1, attrs={"ROL_PROCESAL":"Imputado"})

3) Atributos: Sara  Morales, DNI 35.678.912, con domicilio en 9 de Julio 456…”
   → PER("Sara Morales", group_index=0)
   → DNI("35.678.912", group_index=0)
   → DIRECCION("9 de Julio 456", group_index=0)

4) Medida: “Prohibición de acercamiento del denunciado a la denunciante en un radio de 500 metros.”
   → RELACION("Prohibición de acercamiento del denunciado a la denunciante en un radio de 500 metros",
              attrs={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0,"DISTANCIA_MIN_M":500})

5) Sticky en la misma oración: “la Sra. Ana Lopez, DNI 39112456, con domicilio en calle Belgrano 785…”
   → PER("Ana  Lopez", group_index=0)
   → DNI("39112456", group_index=0)
   → DIRECCION("calle Belgrano 785", group_index=0)

6) Carátula entre comillas + Expte: “… en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.”
   → PER("Rodríguez, Ana Carolina", group_index=0)
   → PER("Fernández, Diego Esteban", group_index=1)
   → NUM_EXPEDIENTE("3187/2023")
""")

In [ ]:
MERGED_PROMPTv2 = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles y ENTIDADES RELEVANTES en documentos judiciales en español.
Leé el documento completo y devolvé SOLO spans EXACTOS (sin parafrasear ni inferir) de las CLASES válidas.
Si una clase NO aparece, no devuelvas nada. No superpongas entidades (una mención = una extracción).

PROHIBICIONES EXPLÍCITAS
- NO extraer palabras sueltas que no sean la entidad en sí (ej.: “DNI” sin número; “fecha” aislada; “Expte.” sin número).
- NO inventar datos, completar por contexto ni normalizar el texto: debe ser el span EXACTO.
- NO inventar clases distintas a las definidas.

FORMATO DE SALIDA
- Debe ser un único JSON válido con la clave "extractions".
- Ejemplo si no hay nada para extraer: {"extractions": []}.

AGRUPAMIENTO (group_index)
- La misma persona (PER) y sus ATRIBUTOS comparten el mismo group_index: DNI, CUIT_CUIL, DIRECCION, TELEFONO,
  CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- Si aparece una PER sin atributos, igual asignale un group_index y usalo en TODAS sus repeticiones.
- Considerá la misma persona aunque cambie el orden (p.ej., “Morales, Julieta Andrea” ≡ “Julieta Andrea Morales”).

REGLA DE AGRUPAMIENTO “STICKY”
- Si un ATRIBUTO personal (DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA)
  aparece a continuación de una PER en la MISMA oración o cláusula (separado por comas o por frases como “DNI”, “con domicilio en”),
  debe usar el MISMO group_index de esa PER.
- No crees un group_index nuevo para un ATRIBUTO si existe una PER inmediatamente precedente en la misma oración.
- Patrones que obligan a “pegar” el atributo a la última PER:
  • "<PER>, DNI <número>" → DNI usa el group_index de esa PER.
  • "<PER>, con domicilio en <span>" → DIRECCION usa el group_index de esa PER.
  • "la Sra./el Sr. <PER> ..." seguido de "DNI ..." o "domicilio ..." en la misma oración → pegar al group_index de esa PER.

ROLES Y ATRIBUTOS
- PER.ROL_PROCESAL solo si está textual o deducible por un patrón claro:
  • “denunciante/denunció” ⇒ {"ROL_PROCESAL":"Denunciante"}
  • “denunciado”, “imputado”, “acusado” ⇒ {"ROL_PROCESAL":"Imputado"}
  • “víctima” ⇒ {"ROL_PROCESAL":"Victima"}
  • “jueza/juez” ⇒ {"ROL_PROCESAL":"Juez"}
  • Otros posibles: "Fiscal","Secretario","Prosecretario","Mediador","Asesor Tutelar","Damnificado","Querellante",
    "Actor","Demandado","Testigo","Perito","Defensor Oficial","Defensor Particular","Apoderado","Tutor","Curador",
    "Policia","Perjudicado","Beneficiario".
- PER.TIPO solo si está textual (ej.: "conyugue_de_la_victima","hijo/a_de_la_victima").
- DIRECCION.TIPO solo si está textual. Valores: "domicilio_real","domicilio_legal","domicilio_laboral",
  "domicilio_de_la_victima","domicilio_del_imputado","domicilio_del_testigo".

REGLAS GENERALES
- Aceptá mayúsculas, tildes faltantes, comillas y formatos no estándar si el span es claro.
- Carátulas/encabezados: en “Apellido, Nombre c/ Apellido, Nombre s/ …” extraé ambos PER con comas y orden exacto,
  aunque estén entre comillas. Si en la misma frase aparece “Expte./Expediente …”, extraé también el NUM_EXPEDIENTE.
- Edades: aceptá “9 años”, “9 años de edad”, “de 9 años” (extraé el núcleo “9 años”) y vinculala al group_index de la persona.
- NUM_EXPEDIENTE: extraé \\d+/\\d{4} cuando aparezca junto a “Expediente”, “Expte.”, “Exp.”, “Exp. N°/Nº/N.o”.
- Si una entidad está mal escrita pero es inequívoca (ej.: en comillas, en carátula), extraela igual.
- No inventes índices para atributos: los atributos personales DEBEN reutilizar el group_index de la última PER mencionada
  en la misma oración/cláusula, salvo que el texto asocie explícitamente ese atributo a otra persona.

VALIDADORES (para evitar falsos positivos)
- DNI: SOLO si tiene 7–8 dígitos o formato ##.###.### (la palabra “DNI” sola NO cuenta).
- CUIT_CUIL: SOLO formato ##-########-#.
- CBU: SOLO 22 dígitos.
- PATENTE_DOMINIO (AR): SOLO AAA123 o AA123AA.
- FECHA: dd/mm/aaaa, dd-mm-aaaa, “dd de <mes> de aaaa”, o compuestas (“5 y 7 de mayo de 2020”).
  No extraer la fecha de resolución del documento ni expresiones vagas (“ayer”, “a las 15:00 horas”).
- LOC: localidades/provincias/países/continentes (no instituciones ni “el domicilio de …”).
- DIRECCION: calle+altura (“9 de Julio 456”), intersección (“Callao y Corrientes”), o número suelto.

RELACION (medidas/vínculos)
- class = RELACION solo si hay una medida o vínculo explícito.
- text = fragmento EXACTO de la medida.
- attrs posibles:
  - "TIPO" ∈ {"prohibicion_acercamiento","cese_perturbacion","contacto_prohibido","orden_cautelar","restriccion_perimetral"}
  - "SUJETO_ACTIVO_GROUP" (int), "SUJETO_PASIVO_GROUP" (int)
  - "DISTANCIA_MIN_M" (int) si está textual
  - "PLAZO" (string corto, ej. "6_meses") si está textual
  - "LUGARES_ALCANZADOS" si está textual

CLASES VÁLIDAS
BANCO, CBU, CORREO_ELECTRONICO, CUIT_CUIL, CUIJ, DIRECCION, DNI, EDAD, ESTUDIOS, FECHA, LINK, LOC, MARCA_AUTOMOVIL,
NACIONALIDAD, NUM_CAJA_AHORRO, NUM_EXPEDIENTE, NUM_MATRICULA, PATENTE_DOMINIO, PER, NUM_ACTUACION, TELEFONO, RELACION.

EJEMPLOS
1) Carátula: "Morales, Julieta Andrea c/ López, Ricardo Daniel s/ Violencia Familiar"
   → PER("Morales, Julieta Andrea", group_index=0)
   → PER("López, Ricardo Daniel", group_index=1)

2) Roles: “la Sra. Julieta Andrea Morales denunció a Ricardo Daniel López…”
   → PER("Julieta Andrea Morales", group_index=0, attrs={"ROL_PROCESAL":"Denunciante"})
   → PER("Ricardo Daniel López", group_index=1, attrs={"ROL_PROCESAL":"Imputado"})

3) Atributos: “Julieta Andrea Morales, DNI 35.678.912, con domicilio en 9 de Julio 456…”
   → PER("Julieta Andrea Morales", group_index=0)
   → DNI("35.678.912", group_index=0)
   → DIRECCION("9 de Julio 456", group_index=0)

4) Medida: “Prohibición de acercamiento del denunciado a la denunciante en un radio de 500 metros.”
   → RELACION("Prohibición de acercamiento del denunciado a la denunciante en un radio de 500 metros",
              attrs={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0,"DISTANCIA_MIN_M":500})

5) Sticky en la misma oración: “la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785…”
   → PER("Ana Carolina Rodríguez", group_index=0)
   → DNI("34.112.456", group_index=0)
   → DIRECCION("calle Belgrano 785", group_index=0)

6) Carátula entre comillas + Expte: “… en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.”
   → PER("Rodríguez, Ana Carolina", group_index=0)
   → PER("Fernández, Diego Esteban", group_index=1)
   → NUM_EXPEDIENTE("3187/2023")
""")

In [ ]:
PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles y ENTIDADES RELEVANTES en documentos judiciales en español.
Devolvés SOLO spans EXACTOS del texto (sin parafrasear). Si una clase no aparece, no devuelvas nada.
No inventes códigos ni completes por contexto. No superpongas entidades (una mención = una extracción).

REGLAS GENERALES
- Aceptá mayúsculas, acentos faltantes, comillas y formatos no estándar si el span es claro.
- Las iniciales de personas cuentan como PER (p. ej., “M.F.R.”).
- Si un nombre aparece con títulos/cargos delante (p. ej., “Sra. Jueza de Familia Dra. Gabriela Montiel”),
  extraé PER solo con el nombre completo (“Gabriela Montiel”) y, si corresponde, agregá ROL_PROCESAL=Juez.
- Si una entidad aparece varias veces, extraela cada vez (spans exactos).

CARÁTULA Y ENCABEZADOS
- En carátulas del tipo: “Apellido, Nombre c/ Apellido, Nombre s/ …” EXTRAÉ:
  • PER: el bloque “Apellido, Nombre” ANTES de “c/”.
  • PER: el bloque “Apellido, Nombre” DESPUÉS de “c/” y ANTES de “s/”.
  Conservá comas y el orden exacto.
- NUM_EXPEDIENTE: extraé \d+/\d{4} cuando aparezca con cualquiera de estas variantes textuales
  (aceptá puntos y espacios variables): “Expediente”, “Expte.”, “Expte”, “Exp.”, “Exp. N°/Nº/N.o”, “Expte. N°/Nº/N.o”.

AGRUPAMIENTO (group_index)
- Uní la misma persona (PER) con sus atributos: DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO,
  EDAD, NACIONALIDAD, NUM_MATRICULA. Ej.: “Ana Carolina Rodríguez, DNI 34.112.456” → mismo group_index.

DIRECCION
- Formatos válidos: calle + altura (“9 de Julio 123”), intersección (“Callao y Corrientes”), o número suelto.
- Atributo DIRECCION.TIPO solo si está explícito (domicilio_real, domicilio_legal, domicilio_laboral, etc.).

LOC
- Localidad / provincia / país, inclusive con prefijos “CIUDAD DE …”, “Ciudad de …”, “Provincia de …”
  y en MAYÚSCULAS. No extraigas instituciones (p. ej., “Juzgado de …”) como LOC.

RELACION (simplificada)
- Usá RELACION solo para medidas/vínculos claros. Sttributes posibles:
  - TIPO ∈ {"prohibicion_acercamiento","cese_perturbacion","contacto_prohibido","orden_cautelar"}
  - SUJETO_ACTIVO_GROUP (int), SUJETO_PASIVO_GROUP (int)
- Mantener spans breves y literales.

CLASES
- PER, DNI (##.###.### o 7–8 dígitos), CUIT_CUIL (##-########-#), CUIJ (formato usual),
  NUM_EXPEDIENTE (\d+/\d{4}), NUM_ACTUACION, DIRECCION, TELEFONO, CORREO_ELECTRONICO,
  LOC, EDAD, NACIONALIDAD, ESTUDIOS, LINK, PATENTE_DOMINIO, MARCA_AUTOMOVIL,
  NUM_CAJA_AHORRO, CBU, NUM_MATRICULA, FECHA (solo fechas explícitas; no “ayer”).
""")


In [6]:
LONG_PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles y de ENTIDADES RELEVANTES para análisis en documentos judiciales en español. 
Leé el documento y extrae SOLO spans exactos (sin parafrasear ni inferir) de las clases definidas más abajo.

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como PER (no confundir con siglas de otras cosas).
- Extraé SOLO spans EXACTOS que estén en el texto.
- Si una clase NO aparece, no devuelvas nada de esa clase.
- No inventes códigos ni números; no completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- Si una entidad está mal escrita, incompleta, repetida o en un formato no estándar, como ser entre comillas o dentro de una carátura, pero claramente corresponde, extraela igual.
- Si una entidad no está clara, no la extraigas.

AGRUPAMIENTO:
- Cada persona (PER) se agrupa con sus atributos en el mismo group_index: DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- En PER podés usar atributo ROL_PROCESAL solo si está explícito en el texto. 
   Valores posibles de ROL_PROCESAL:
   "Juez", "Fiscal", "Secretario", "Prosecretario", "Mediador", "Asesor Tutelar", "Imputado", "Acusado", 
   "Victima", "Damnificado", "Denunciante", "Querellante", "Actor", "Demandado", "Testigo", 
   "Perito", "Defensor Oficial", "Defensor Particular", "Apoderado", "Tutor", "Curador", "Policia", 
   "Perjudicado", "Beneficiario".
- En DIRECCION podés usar atributo TIPO solo si está indicado en el texto. Valores posibles:
   "domicilio_real", "domicilio_legal", "domicilio_laboral", "domicilio_de_la_victima", "domicilio_del_imputado","domicilio_del_testigo"
- En PER podés usar atributo TIPO solo si está indicado en el texto. Valores posibles:
   "conyugue_de_la_victima", "hijo/a_de_la_victima", etc
- Una DIRECCION puede expresarse como: 
   - calle + altura ("9 de Julio 123"),
   - intersección de calles ("Callao y Corrientes"),
   - número de domicilio aislado.

RELACIONES INTER-GRUPOS:
- Usá la clase RELACION para conectar grupos de personas.
- extraction_text = fragmento exacto de la medida o vínculo.
- attributes posibles:
   - TIPO: "prohibicion_acercamiento", "cese_perturbacion", "orden_cautelar", "contacto_prohibido", "restriccion_perimetral".
   - SUJETO_ACTIVO_GROUP: id del acusado/imputado.
   - SUJETO_PASIVO_GROUP: id de la víctima/denunciante/damnificado.
   - PLAZO: duración si está explícita (ejemplo: "6_meses").
   - DISTANCIA_MIN_M: distancia mínima en metros si está explícita (ejemplo: 500).
   - LUGARES_ALCANZADOS: direcciones o expresiones exactas como "cualquier lugar donde se encuentre la víctima".

CLASES:
- BANCO: entidad bancaria (no vale solo “Banco”).
- CBU: número de 22 dígitos.
- CORREO_ELECTRONICO: email (no el del juzgado).
- CUIT_CUIL: ##-########-#.
- CUIJ: código judicial ##-########-#.
- DIRECCION: domicilio en cualquiera de las formas listadas arriba.
- DNI: 7-8 dígitos o ##.###.### (no solo la palabra “DNI”).
- EDAD: edad en años o meses.
- ESTUDIOS: nivel educativo (primario, secundario, terciario, universitario, posgrado, doctorado; puede incluir “incompleto”, “completo”, “finalizado”, “en curso”).
- FECHA: fechas explícitas (excepto la de resolución del documento, y no expresiones vagas como “ayer” o “a las 15:00 horas”).
- LINK: URL.
- LOC: localidad, provincia, país, continente (no hospitales ni referencias genéricas).
- MARCA_AUTOMOVIL: marca de vehículo.
- NACIONALIDAD: nacionalidad.
- NUM_CAJA_AHORRO: número de caja de ahorro/cuenta.
- NUM_EXPEDIENTE: \d+/\d{4}.
- NUM_MATRICULA: matrícula profesional o académica.
- PATENTE_DOMINIO: dominio de vehículo (AAA123 o AA123AA).
- PER: nombre completo, iniciales o apodo de persona física.
- NUM_ACTUACION: número de actuación administrativa/contravencional.
- TELEFONO: fijo o celular.
- RELACION: vínculo o medida entre personas (ver atributos arriba).
""")


In [ ]:
# Si quisieramos excluir direcciones o ciertos campos agregar al prompt por ejemplo:
excepcion_prompt_comment = """- No extraigas:
   1) La fecha y lugar de la resolución del documento (ejemplo: "Buenos Aires, 29 de julio de 2022"), que suele aparecer al comienzo.
   2) Información del juzgado como mail, dirección, teléfono o redes sociales (ejemplo: "Juzgado PCyF No 10 - Tacuarí 138, 7o Piso - juzcyf10@jusbaires.gob.ar - 4014-6821/20 - @jpcyf10"), que suele estar al pie del documento.
"""

In [ ]:
NO_ATTRIBUTES_PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles para anonimización en documentos judiciales en español. Vas a leer cada parrafo con atención y extraer todas las entidades que correspondan según
las clases definidas más abajo. 

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como clase Persona, no confundir con siglas de otras cosas.
- Extraé SOLO spans EXACTOS que estén en el texto (no parafrasees ni infieras).
- Si una clase NO aparece, NO devuelvas nada de esa clase.
- NO inventes códigos ni números. No completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- NO son sensibles las siguientes entidades: 
                         1. La fecha y lugar de la resolucion del documento (por ej. "Buenos Aires, 29 de julio de 2022"), suele aparecen al comienzo del documento.
                         2. Información del juzgado como mail, direccion, telefono y cuenta de red social (por ej. "'Juzgado PCyF No 10 - Tacuarí 138, 7o Piso - juzcyf10@jusbaires.gob.ar - 4014-6821/20 - @jpcyf10'"), suele estar al pie del documento.
                         3. Las personas no sensibles como jueces, fiscales y secretarios.

- Si una entidad no está clara, NO la extraigas.
- Si una entidad está sutilmente mal escrita, incompleta, repetida o en un formato no estándar, pero detectas que corresponde a esa entendidad, extráela igual.

POSIBLES CLASES Y DESCRIPCIONES:
- BANCO: Debe especificar una entidad bancaria, 'Banco' no aplica como tal.
- CBU: número de 22 dígitos asociado a una entidad bancaria
- CORREO_ELECTRONICO: dirección de email, que no sea del juzgando.
- CUIT_CUIL: código único de identificación tributaria o laboral en Argentina (formato ##-########-#)
- CUIJ: código único de identificación judicial (formato ##-########-#)
- DIRECCION: puede presentarse como calle y altura (por ej. "9 de Julio 123"), intersección de calles (por ej. "Callao y Corrientes"), o número de domicilio 
- DNI: documento nacional de identidad de 7-8 dígitos, puede presentarse en el formato ##.###.### (numeros separados con puntos), la expresión "DNI" sin número no aplica como DNI.
- EDAD: edad de una persona, puede estar en años o meses
- ESTUDIOS: nivel educativo alcanzado (primario, secundario, terciario, universitario, posgrado, doctorado), puede estar acompañado de "incompleto", "completo", "finalizado", "en curso"
- FECHA: fechas en cualquier formato (dd/mm/aaaa, dd-mm-aaaa, dd de mes de aaaa, también puede ser dos fechas juntas como por ejemplo el 5 y 7 de mayo de 2020 y similares). No asignar como FECHA la fecha de resolución del documento, tampoco expresiones de fechas que no refieren a una fecha en particular, como por ejemplo, "en el día de ayer" o "a las 15:00 horas"
- LINK: URLs o enlaces web.
- LOC: nombres de localidades, provincias, países, continentes. NO asignar como LOC a lugares que no sean locaciones, como ser hospitales o referencias a lugares por nombres como "el domicilio de Olavarria"
- MARCA_AUTOMOVIL: marcas de automóviles (Ford, Chevrolet, Toyota, Renault, Fiat, etc)
- NACIONALIDAD: nacionalidades (argentina, italiana, española, uruguaya, chilena, paraguara, etc)
- NUM_CAJA_AHORRO: número de caja de ahorro o cuenta bancaria
- NUM_EXPEDIENTE: número de expediente judicial o administrativo en formato \d+/\d{4} (por ejemplo 1234/2020)
- NUM_MATRICULA: número de matrícula profesional (médica, abogacía, etc) o académica.
- PATENTE_DOMINIO: patentes o dominio de un vehículo. En Argentina, pueden ser de formato [A-Z]{3}\d{3} o [A-Z]{2}\d{3}[A-Z]{2}
- PER: Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.                     
- NUM_ACTUACION: Número identificatorio de una actuación administrativa o contravencional.
- TELEFONO: Número telefónico (fijo o celular).

""")


In [7]:
examples = [

# 1) PER/FECHA/DIRECCION/LOC (carátula + encabezado + dirección + fecha)
lx.data.ExampleData(
    text=textwrap.dedent("""JUZGADO DE FAMILIA N.º 1 DE LA CIUDAD DE SAN LORENZO
Expediente N.º 3187/2023
Carátula: Rodríguez, Carla Carolina c/ Fernández, Carlos Esteban s/ Violencia Familiar
SENTENCIA
En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, la Sra. Jueza Dra. Verónica Perez dicta resolución.
Con fecha 10 de noviembre de 2023, la Sra. Carla Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, San Lorenzo, denunció al Sr. Carlos Esteban Fernández, DNI 31.998.210."""),
    extractions=[
        lx.data.Extraction(extraction_class="PER", extraction_text="Rodríguez, Carla Carolina", group_index=0),
        lx.data.Extraction(extraction_class="PER", extraction_text="Fernández, Carlos Esteban", group_index=1),
        lx.data.Extraction(extraction_class="LOC",  extraction_text="CIUDAD DE SAN LORENZO"),
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="3187/2023"),
        lx.data.Extraction(extraction_class="LOC",  extraction_text="San Lorenzo"),
        lx.data.Extraction(extraction_class="LOC",  extraction_text="Provincia de Santa Fe"),
        lx.data.Extraction(extraction_class="PER",  extraction_text="Verónica Perez", group_index=2, attributes={"ROL_PROCESAL":"Juez"}),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="22 días del mes de noviembre de 2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="10 de noviembre de 2023"),
        lx.data.Extraction(extraction_class="PER",       extraction_text="Carla Carolina Rodríguez", group_index=0),
        lx.data.Extraction(extraction_class="DNI",       extraction_text="34.112.456",            group_index=0),
        lx.data.Extraction(extraction_class="DIRECCION", extraction_text="calle Belgrano 785, San Lorenzo", group_index=0),
        lx.data.Extraction(extraction_class="PER", extraction_text="Carlos Esteban Fernández", group_index=1),
        lx.data.Extraction(extraction_class="DNI", extraction_text="31.998.210",             group_index=1),
    ],
),

# 2) Único ejemplo de códigos/formatos + vehículo en narración
lx.data.ExampleData(
    text=textwrap.dedent("""CUIJ: IPP J-01-00017381-5/2021-0
Actuación Nro: 14544192/2021
Expte. N.o 4533/2024
El imputado se retiró manejando un vehículo Volkswagen Voyage, dominio KXY-876, por la Av. Corrientes. El video pertinente a la causa se encuentra disponible en https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv, grabado el 4 de abril de 2021"""),
    extractions=[
        lx.data.Extraction(extraction_class="CUIJ",              extraction_text="IPP J-01-00017381-5/2021-0"),
        lx.data.Extraction(extraction_class="NUM_ACTUACION",     extraction_text="14544192/2021"),
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE",    extraction_text="4533/2024"),
        lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL",   extraction_text="Volkswagen Voyage"),
        lx.data.Extraction(extraction_class="PATENTE_DOMINIO",   extraction_text="KXY-876"),
        lx.data.Extraction(extraction_class="LINK",              extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),
        lx.data.Extraction(extraction_class="FECHA",             extraction_text="4 de abril de 2021"),
    ],
),

# 3) Datos personales típicos + otro vehículo/patente en frase fluida
lx.data.ExampleData(
    text=textwrap.dedent("""El Fiscal Carlos Garcia solicitó medidas respecto de ROBERTO CARUZO, DNI 30.112.642, 34 años de edad, nacionalidad paraguaya, con estudios secundarios completos, teléfono 1141504528 y correo electrónico rob.caruzo@gmail.com. 
En ese momento conducía un Ford Focus AA123BB por la Av. Rivadavia, en dirección a Caballito. El acusado violentó a Maria Garcia, de 25 años y DNI 45091234"""),
    extractions=[
        lx.data.Extraction(extraction_class="PER", extraction_text="Carlos Garcia", attributes={"ROL_PROCESAL":"Fiscal"}, group_index=0),
        lx.data.Extraction(extraction_class="PER",              extraction_text="ROBERTO CARUZO",           group_index=1, attributes={"ROL_PROCESAL": "Acusado"}),
        lx.data.Extraction(extraction_class="DNI",              extraction_text="30.112.642",               group_index=1),
        lx.data.Extraction(extraction_class="EDAD",             extraction_text="34",                       group_index=1),
        lx.data.Extraction(extraction_class="NACIONALIDAD",     extraction_text="paraguaya",                group_index=1),
        lx.data.Extraction(extraction_class="ESTUDIOS",         extraction_text="estudios secundarios completos", group_index=1),
        lx.data.Extraction(extraction_class="TELEFONO",         extraction_text="1141504528",               group_index=1),
        lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="rob.caruzo@gmail.com",  group_index=1),
        lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL",  extraction_text="Ford Focus",               group_index=1),
        lx.data.Extraction(extraction_class="PATENTE_DOMINIO",  extraction_text="AA123BB",                  group_index=1),
        lx.data.Extraction(extraction_class="PER",              extraction_text="Maria Garcia",               group_index=3, attributes={"ROL_PROCESAL": "Victima"}),
        lx.data.Extraction(extraction_class="DNI",              extraction_text="45091234",               group_index=3),
        lx.data.Extraction(extraction_class="EDAD",             extraction_text="25",                       group_index=3),       
    ],
),

# 4) NEGATIVO
lx.data.ExampleData(
    text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
    extractions=[],
),
]


In [8]:
# -----------------
# Ejemplos balanceados (judiciales)
#   1) PER/FECHA/DIRECCION/LOC
#   2) Un único ejemplo de códigos (para enseñar formato)
#   3) Datos personales típicos de actuaciones
#   4) NEGATIVO: no hay códigos -> salida vacía
# -----------------
long_examples = [

lx.data.ExampleData(
    text=textwrap.dedent("""JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar
SENTENCIA
En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.
I. ANTECEDENTES
Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial.
La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó a aislarla de su entorno familiar, controlar sus gastos y proferir insultos y amenazas, que en los últimos meses derivaron en empujones y golpes.
Se acompañaron constancias médicas del Hospital "San Martín" de fechas 14/08/2023 y 08/11/2023, donde se registran lesiones en antebrazos y región lumbar, así como un informe psicológico que describe un cuadro de depresión moderada y ansiedad.
II. MEDIDAS CAUTELARES ADOPTADAS
Con fecha 11 de noviembre de 2023, este Juzgado dispuso:
Exclusión inmediata del denunciado del domicilio conyugal.
Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros.
Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales.
III. PRUEBA PRODUCIDA
En audiencia celebrada el 17 de noviembre de 2023, declararon la Sra. Patricia Gómez, vecina del domicilio conyugal, y el Sr. Luis Alberto Rivas, compañero de trabajo de la denunciante, quienes relataron haber presenciado episodios de gritos, discusiones y amenazas por parte del denunciado.
El Equipo Interdisciplinario del Juzgado emitió informe en el que concluyó que existe un riesgo alto de reiteración de la violencia, con impacto negativo en el bienestar emocional de la menor.
IV. FUNDAMENTOS
De la valoración integral de la prueba surge acreditada la existencia de violencia física, psicológica y patrimonial ejercida por el Sr. Diego Esteban Fernández contra la Sra. Ana Carolina Rodríguez, en el marco de una relación de pareja y con afectación a la hija menor.
La Ley Nacional 26.485 y la Ley Provincial 11.529 obligan a adoptar medidas urgentes y eficaces para proteger a las víctimas de violencia de género.
En este caso, la reiteración de los hechos, la proximidad de los domicilios y la vulnerabilidad de la menor justifican la extensión de las medidas cautelares y la adopción de acciones complementarias.
V. RESUELVO
Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez y de la hija menor M.F.R.
Mantener la exclusión del denunciado del domicilio conyugal. La grabación de la sentencia se encuentra disponible en el link: https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv. """),
    extractions=[
        # Personas y atributos
        # Carátula: nombres invertidos
        lx.data.Extraction(extraction_class="PER", extraction_text="Rodríguez, Ana Carolina", group_index=0, attributes={"ROL_PROCESAL":"Denunciante"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Fernández, Diego Esteban", group_index=1),

        # Repetición de expediente
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="3187/2023"),

        # Relación de cónyuge / atributo
        lx.data.Extraction(extraction_class="PER", extraction_text="Diego Esteban Fernández", group_index=1, attributes={"RELACION_CON_VICTIMA":"cónyuge"}),

        # Relación de hija
        lx.data.Extraction(extraction_class="PER", extraction_text="M.F.R.", group_index=2, attributes={"RELACION_CON_VICTIMA":"hija"}),

        # Roles de testigos más contexto
        lx.data.Extraction(extraction_class="PER", extraction_text="Patricia Gómez", group_index=4, attributes={"ROL_PROCESAL":"Testigo","RELACION_CON_VICTIMA":"vecina"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Luis Alberto Rivas", group_index=5, attributes={"ROL_PROCESAL":"Testigo","RELACION_CON_VICTIMA":"compañero de trabajo"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Ana Carolina Rodríguez", group_index=0, attributes={"ROL_PROCESAL":"Denunciante"}),
        lx.data.Extraction(extraction_class="DNI", extraction_text="34.112.456", group_index=0),
        lx.data.Extraction(extraction_class="DIRECCION", extraction_text="calle Belgrano 785, Barrio Centro, San Lorenzo", group_index=0, attributes={"TIPO":"domicilio_de_la_victima"}),

        lx.data.Extraction(extraction_class="PER", extraction_text="Diego Esteban Fernández", group_index=1),
        lx.data.Extraction(extraction_class="DNI", extraction_text="31.998.210", group_index=1),
        lx.data.Extraction(extraction_class="DIRECCION", extraction_text="calle Moreno 1520", group_index=1, attributes={"TIPO":"domicilio_del_imputado"}),

        lx.data.Extraction(extraction_class="PER", extraction_text="M.F.R.", group_index=2),
        lx.data.Extraction(extraction_class="EDAD", extraction_text="9 años", group_index=2),

        lx.data.Extraction(extraction_class="PER", extraction_text="Verónica Salvatierra", group_index=3, attributes={"ROL_PROCESAL":"Juez"}),

        lx.data.Extraction(extraction_class="PER", extraction_text="Patricia Gómez", group_index=4, attributes={"ROL_PROCESAL":"Testigo"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Luis Alberto Rivas", group_index=5, attributes={"ROL_PROCESAL":"Testigo"}),

        # Localidades
        lx.data.Extraction(extraction_class="LOC", extraction_text="San Lorenzo"),
        lx.data.Extraction(extraction_class="LOC", extraction_text="Provincia de Santa Fe"),

        # Expediente
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="3187/2023"),

        # Fechas (excluye la de resolución)
        lx.data.Extraction(extraction_class="FECHA", extraction_text="10 de noviembre de 2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="14/08/2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="08/11/2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="11 de noviembre de 2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="17 de noviembre de 2023"),

        #Link
        lx.data.Extraction(extraction_class="LINK", extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),

        # Relaciones (medidas cautelares y vínculos)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros.",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0,"DISTANCIA_MIN_M":500}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros.",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2,"DISTANCIA_MIN_M":500}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales.",
            attributes={"TIPO":"contacto_prohibido","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales.",
            attributes={"TIPO":"contacto_prohibido","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Exclusión inmediata del denunciado del domicilio conyugal.",
            attributes={"TIPO":"orden_cautelar","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0,"PLAZO":"180_dias"}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la hija menor M.F.R.",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2,"PLAZO":"180_dias"}
        ),

        # Vínculo conyugal / convivencial (entre personas)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="su cónyuge",
            attributes={"TIPO":"vinculo_conyugal","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="conviven desde hace doce años",
            attributes={"TIPO":"vinculo_convivencia","SUJETO_ACTIVO_GROUP":0,"SUJETO_PASIVO_GROUP":1}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="tienen una hija en común, M.F.R.",
            attributes={"TIPO":"vinculo_filiacion","SUJETO_ACTIVO_GROUP":0,"SUJETO_PASIVO_GROUP":2}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="tienen una hija en común, M.F.R.",
            attributes={"TIPO":"vinculo_filiacion","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2}
        ),

    ],
),
lx.data.ExampleData(
    text=textwrap.dedent("""
        JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19
        GOMEZ, ELVIS JUNIOR SOBRE 89 - LESIONES LEVES
        Número: IPP 8125/2020-0
        CUIJ: IPP J-01-00017381-5/2021-0
        Actuación Nro: 14544192/2021
        ACTA DE AUDIENCIA
        VIDEOCONFERENCIA
        "GOMEZ, ELVIS JUNIOR SOBRE 89 EN FUNCIÓN DEL 92, 149 BIS, 162 Y 239 DEL CÓDIGO PENAL"
        Causa N° 8122/2021
        Fecha: 4 de abril de 2021
        Horario de inicio: 12:00 horas
        Tipo de audiencia: audiencia de conocimiento personal (art. 266 CPPCABA)
        Juez: Pablo C. Casas -Juzgado Penal Contravencional y de Faltas Nro. 10-.
        Secretaria: Maria Agustina Iriarte López.
        PARTES PRESENTES
        Acusado: Carlos Junior PEREZ, DNI n° 50.966.533.
        Defensa Oficial: Marina Recabarra, -Defensoría Oficial Nro. 20-.
        Fiscal: Adrián Dávila -Fiscalía Penal, Contravencional y de Faltas Nro. 36-.
        DESARROLLO
        El 15 de marzo de 2021, alrededor de las 23:00 horas, mientras se encontraba en una reunión en la casa de una señora llamada SOL,
        en Villa Pueyrredón de esta ciudad, se puso agresivo con su pareja BELEN GUTIERREZ, le pegó dos piñas en la cara,
        la agarró del cuello y la tiró al piso. Luego procedió a llevarse las llaves de su vehículo Volkswagen Voyage, dominio KXY-876."""),
            
    extractions=[
        # Identificadores judiciales
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="8122/2021"),
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="8125/2020"),
        lx.data.Extraction(extraction_class="CUIT_CUIL", extraction_text="IPP J-01-00017381-5/2021-0"),
        lx.data.Extraction(extraction_class="NUM_ACTUACION", extraction_text="14544192/2021"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="4 de abril de 2021"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="15 de marzo de 2021"),
        lx.data.Extraction(extraction_class="LOC", extraction_text="Villa Pueyrredón"),

        # Funcionario judicial
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Pablo C. Casas",
            attributes={"ROL_PROCESAL": "Juez"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Maria Agustina Iriarte López",
            attributes={"ROL_PROCESAL": "Secretario"},
            group_index=2
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Adrián Dávila",
            attributes={"ROL_PROCESAL": "Fiscal"},
            group_index=3
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Marina Recabarra",
            attributes={"ROL_PROCESAL": "Defensor Oficial"},
            group_index=4
        ),

        # Acusado
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Carlos Junior PEREZ",
            attributes={"ROL_PROCESAL": "Acusado"},
            group_index=5
        ),
        lx.data.Extraction(
            extraction_class="DNI",
            extraction_text="50.966.533",
            group_index=5
        ),

        # Víctima
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="BELEN GUTIERREZ",
            attributes={"ROL_PROCESAL": "Victima"},
            group_index=6
        ),
        lx.data.Extraction(extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876",group_index=6),
        lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL", extraction_text="Volkswagen Voyage",group_index=6),
        # Testigo circunstancial (persona mencionada, no procesal)
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="SOL",
            attributes={"ROL_PROCESAL": "Testigo"},
            group_index=7
        ),

        # Relación entre acusado y víctima (violencia física)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="le pegó dos piñas en la cara, la agarró del cuello y la tiró al piso",
            attributes={
                "TIPO": "cese_perturbacion",   # violencia física → medida/restricción
                "SUJETO_ACTIVO_GROUP": 5,
                "SUJETO_PASIVO_GROUP": 6
            }
        )
    ],
),
lx.data.ExampleData(
    text=textwrap.dedent("""
        Hoy el Fiscal Carlos Garcia solicitó que se ordene a ROBERTO CARUZO, DNI 30.112.642, 34 años de edad, nacionalidad paraguaya, con 
        estudios secundarios completos, por el plazo de seis meses, el cese en los actos de perturbación o intimidación que, directa o indirectamente, 
        realice hacia la persona de la damnificada BELEN CASIO, CUIL 27-37.412.987-6; 2) por el plazo de seis (6) meses, se imponga a ROBERTO CARUZO, DNI 30.112.642
        la prohibición de acercamiento y contacto hacia la víctima BELEN CASIO, 37.412.987, –en el lugar que se encuentre– de
        modo que deberá suspender todo tipo de contacto físico y/o por cualquier medio que signifique intromisión injustificada
        en relación a la persona de la damnificada, por sí o por intermedio de terceras personas y la prohibición de acercamiento
        a menos de 500 metros de los domicilios ubicados en Av. Pedro Goyena 51, piso 7° dpto. "A", y Callao 543, de esta Ciudad. Se requirió  informes  al  
        Banco BBVA   Francés, respecto  de  las  cuentas  bancarias  del denunciado ROBERTO CARUZO, identificadas  como  Caja  de  ahorro  en  pesos  argentinos 
        número  117-59824/6 con  CBU  0180132640000004685591, teléfono celular 1141504528 y dirección de correo electrónico rob.caruzo@gmail.com."""),
    extractions=[

        # Fiscal (grupo 0: funcionario, si querés diferenciarlo)
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Carlos Garcia",
            attributes={"ROL_PROCESAL": "Fiscal"},
            group_index=0
        ),
        # Víctima
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="BELEN CASIO",
            attributes={"ROL_PROCESAL": "Victima"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="CUIL",
            extraction_text="27-37.412.987-6",
            group_index=1
        ),
        # Direcciones de la víctima (ambas al grupo 1)
        lx.data.Extraction(
            extraction_class="DIRECCION",
            extraction_text="Av. Pedro Goyena 51, piso 7° dpto. \"A\"",
            attributes={"TIPO": "domicilio_de_la_victima"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="DIRECCION",
            extraction_text="Callao 543",
            attributes={"TIPO": "domicilio_de_la_victima"},
            group_index=1
        ),

        # Acusado / Imputado
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="ROBERTO CARUZO",
            attributes={"ROL_PROCESAL": "Acusado"},
            group_index=2
        ),
        lx.data.Extraction(
            extraction_class="DNI",
            extraction_text="30.112.642",
            group_index=2
        ),
        lx.data.Extraction(extraction_class="BANCO",         extraction_text="Banco BBVA", group_index=2),
        lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6", group_index=2),
        lx.data.Extraction(extraction_class="CBU",        extraction_text="0180132640000004685591", group_index=2),
        lx.data.Extraction(extraction_class="TELEFONO",     extraction_text="1141504528",group_index=2),
        lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="rob.caruzo@gmail.com",group_index=2),
        lx.data.Extraction(extraction_class="EDAD",        extraction_text="34", group_index=2),
        lx.data.Extraction(extraction_class="NACIONALIDAD",extraction_text="paraguaya", group_index=2),
        lx.data.Extraction(extraction_class="ESTUDIOS",    extraction_text="estudios secundarios completos", group_index=2),

        # Relación / Medidas (conecta grupos 2 -> 1)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="cese en los actos de perturbación o intimidación por el plazo de seis meses",
            attributes={
                "TIPO": "cese_perturbacion",
                "SUJETO_ACTIVO_GROUP": 2,
                "SUJETO_PASIVO_GROUP": 1,
                "PLAZO": "6_meses"
            }
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="prohibición de acercamiento y contacto por el plazo de seis (6) meses a menos de 500 metros",
            attributes={
                "TIPO": "prohibicion_acercamiento",
                "SUJETO_ACTIVO_GROUP": 2,
                "SUJETO_PASIVO_GROUP": 1,
                "PLAZO": "6_meses",
                "DISTANCIA_MIN_M": 500,
                "LUGARES_ALCANZADOS": [
                    "Av. Pedro Goyena 51, piso 7° dpto. \"A\"",
                    "Callao 543",
                    "cualquier lugar donde se encuentre la víctima"
                ]
            }
        ),
    ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. "
              "No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
        extractions=[],  # ejemplo negativo: desalienta devolver clases ausentes
    ),
]

## Functions

In [9]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs

def langextract_to_dict(result):
    out = []
    for ext in getattr(result, "extractions", []) or []:
        label = getattr(ext, "extraction_class", None)
        text = getattr(ext, "extraction_text", None)

        ci = getattr(ext, "char_interval", None)
        start_char = getattr(ci, "start_pos", None) if ci else None
        end_char = getattr(ci, "end_pos", None) if ci else None

        attrs = getattr(ext, "attributes", {}) or {}
        alignment_status = getattr(ext, "alignment_status", None)  # ← FIX

        out.append({
            "label": label,
            "text": text,
            "start_char": start_char,
            "end_char": end_char,
            "attrs": attrs,
            "alignment_status": alignment_status
        })
    return out


def langextract_prediction(text,PROMPT, examples,openai_api_key):

    result =  lx.extract(
        text_or_documents=text,
        prompt_description=PROMPT,
        examples=examples,
        language_model_type=lx.inference.OpenAILanguageModel,
        model_id="gpt-4o",
        api_key=openai_api_key,
        max_char_buffer=1000, #1000,
        extraction_passes=2, #1,
        max_workers=4, #6,
        fence_output=True,
        use_schema_constraints=False, # https://github.com/google/langextract
        language_model_params={
            "temperature": 0.0, #0.1,
            "top_p": 1.0,
            "max_tokens": 400,
            "timeout": 600,},
            debug=False)
    return result, langextract_to_dict(result)

def sample_cases(df, label, model="prediction", n=3):
    # nombre dinámico para el campo de predicción principal
    pred_key = "openai" if model == "prediction" else ("ner" if model == "NER_prediction" else "predmodel")

    def _norm_label_text(item):
        if not isinstance(item, dict):
            return None, ""
        attrs = item.get("attrs", {}) or {}
        lab   = attrs.get("aymurai_label") or item.get("label")
        txt   = attrs.get("aymurai_alt_text") or item.get("extraction_text") or item.get("text", "")
        return lab, txt

    cases = {"TP": [], "FP": [], "FN": []}

    for _, row in df.iterrows():
        # validation
        val_raw = eval(row["validation"]) if row["validation"] else []
        val_spans = [_norm_label_text(v) for v in val_raw if isinstance(v, dict)]
        val_spans = [(lab, txt) for lab, txt in val_spans if lab is not None]

        # pred principal (según 'model')
        if model == "NER_prediction":
            pred_raw = eval(row[model]) if row.get(model) else []
        else:
            pred_raw = row.get(model) if row.get(model) else []
        pred_spans = [_norm_label_text(p) for p in pred_raw if isinstance(p, dict)]
        pred_spans = [(lab, txt) for lab, txt in pred_spans if lab is not None]

        # pred de ambos modelos (si existen las columnas)
        openai_raw = row.get("prediction")
        ner_raw    = row.get("NER_prediction")
        openai_list = openai_raw if (openai_raw and not isinstance(openai_raw, str)) else (eval(openai_raw) if openai_raw else [])
        ner_list    = ner_raw if (ner_raw and not isinstance(ner_raw, str)) else (eval(ner_raw) if ner_raw else [])

        openai_spans = [_norm_label_text(p) for p in (openai_list or []) if isinstance(p, dict)]
        openai_spans = [(lab, txt) for lab, txt in openai_spans if lab is not None]
        ner_spans    = [_norm_label_text(p) for p in (ner_list or []) if isinstance(p, dict)]
        ner_spans    = [(lab, txt) for lab, txt in ner_spans if lab is not None]

        # listas planas por label
        val_labels  = [lab for lab, _ in val_spans]
        pred_labels = [lab for lab, _ in pred_spans]

        # snippet e info extra
        text     = (row.get("text_x") or "")#[:300]
        para_id  = row.get("paragraph_id", None)
        doc_name = row.get("name", None)

        # armar registro con val + ambas preds + clave dinámica
        base_rec = {
            "paragraph_id": para_id,
            "document": doc_name,
            "parrafo": text,
            "val":   [t for lab, t in val_spans if lab == label],
            "openai": [t for lab, t in openai_spans if lab == label],
            "ner":    [t for lab, t in ner_spans    if lab == label],
        }
        # setear pred principal bajo su clave dinámica
        base_rec[pred_key] = [t for lab, t in pred_spans if lab == label]

        # clasificar TP/FP/FN
        if (label in val_labels) and (label in pred_labels):
            cases["TP"].append(base_rec)
        elif (label in val_labels) and (label not in pred_labels):
            cases["FN"].append(base_rec)
        elif (label not in val_labels) and (label in pred_labels):
            cases["FP"].append(base_rec)

    # muestra aleatoria top-n por tipo
    return {k: random.sample(v, min(len(v), n)) for k, v in cases.items() if v}

def _norm(s: str) -> str:
    # normaliza nombre para comparar: minúsculas, sin acentos, sin signos, sin comillas
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = re.sub(r'["“”]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s

def _swap_name(s: str) -> str:
    # genera variante "Nombre Apellido" <-> "Apellido, Nombre"
    if "," in s:
        ap, nom = [p.strip() for p in s.split(",", 1)]
        return f"{nom} {ap}"
    parts = s.split()
    if len(parts) >= 2:
        # heurística simple: último token como apellido
        return f"{parts[-1]}, {' '.join(parts[:-1])}"
    return s

def dedupe_and_fix(extractions, PERSON_ATTRS = False):
    # 1) Filtrar clases inválidas
    cleaned = [e for e in extractions if e.extraction_class in VALID_CLASSES]

    # 2) Asignar group_index consistente para PER y sus atributos
    #    Mapeo canónico por nombre normalizado (incluyendo variante permutada)
    name2gid = {}
    next_gid = 0

    # primera pasada: asignar gid a cada PER por texto exacto (con variantes)
    for e in cleaned:
        if e.extraction_class == "PER":
            t = e.extraction_text
            k1 = _norm(t)
            k2 = _norm(_swap_name(t))
            gid = None
            if k1 in name2gid:
                gid = name2gid[k1]
            elif k2 in name2gid:
                gid = name2gid[k2]
            else:
                gid = next_gid
                name2gid[k1] = gid
                name2gid[k2] = gid
                next_gid += 1
            e.group_index = gid

    # segunda pasada: propagar gid a atributos cercanos de persona si no lo tienen
    # estrategia simple: para cada atributo de persona, tomar el PER más cercano por distancia de caracteres
    person_spans = [(e.group_index, e.char_interval.start_pos, e.char_interval.end_pos)
                    for e in cleaned if e.extraction_class == "PER" and e.char_interval]
    if not PERSON_ATTRS:
        PERSON_ATTRS = {"DNI","CUIT_CUIL","DIRECCION","TELEFONO","CORREO_ELECTRONICO","EDAD","NACIONALIDAD","NUM_MATRICULA"}
    for e in cleaned:
        if e.extraction_class in PERSON_ATTRS:
            if e.group_index is not None:
                continue
            if e.char_interval is None:
                continue
            s = e.char_interval.start_pos
            # buscar PER más cercano
            best = None
            best_dist = 10**9
            for gid, ps, pe in person_spans:
                if ps is None or pe is None:
                    continue
                # distancia mínima al span
                dist = min(abs(s-ps), abs(s-pe))
                if dist < best_dist:
                    best_dist = dist
                    best = gid
            if best is not None:
                e.group_index = best

    # 3) Reindexar GIDs a 0..K-1 en orden de aparición (opcional, para estética)
    #    (solo si querés que queden compactos)
    gid_map = {}
    ng = 0
    for e in cleaned:
        if e.group_index is not None:
            if e.group_index not in gid_map:
                gid_map[e.group_index] = ng
                ng += 1
            e.group_index = gid_map[e.group_index]

    return cleaned


### Sanitize

In [10]:
import json, unicodedata, re
from typing import List

ALLOWED = {
  "BANCO","CBU","CORREO_ELECTRONICO","CUIT_CUIL","CUIJ","DIRECCION","DNI","EDAD","ESTUDIOS","FECHA",
  "LINK","LOC","MARCA_AUTOMOVIL","NACIONALIDAD","NUM_CAJA_AHORRO","NUM_EXPEDIENTE","NUM_MATRICULA",
  "PATENTE_DOMINIO","PER","NUM_ACTUACION","TELEFONO","RELACION"
}
PERSON_ATTRS = {"DNI","CUIT_CUIL","DIRECCION","TELEFONO","CORREO_ELECTRONICO","EDAD","NACIONALIDAD","NUM_MATRICULA"}

def _norm_person(s: str) -> str:
    s = s.strip()
    if "," in s:
        ap, no = [x.strip() for x in s.split(",", 1)]
        s = f"{no} {ap}"
    s = " ".join(s.split()).lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c)!="Mn")
    return s

def sanitize_and_fold(extractions: List, full_text: str):
    # 0) Pliegue de "attributes" y "group_index" fantasmas sobre el ítem anterior con mismo group_index
    by_gid_last_item = {}
    cleaned = []
    for ex in extractions:
        cls = getattr(ex, "extraction_class", None)
        gid = getattr(ex, "group_index", None)
        if cls in {"attributes","attrs","group_index","extraction_text"}:
            # Intentar parsear dict de attributes si existe y plegarlo
            try:
                payload = ex.extraction_text
                if isinstance(payload, str) and payload.strip():
                    attrs = json.loads(payload.replace("'", '"'))
                else:
                    attrs = {}
            except Exception:
                attrs = {}
            if gid is not None and gid in by_gid_last_item and attrs:
                tgt = by_gid_last_item[gid]
                tgt.attributes = {**(tgt.attributes or {}), **attrs}
            continue
        # normal item
        cleaned.append(ex)
        if gid is not None:
            by_gid_last_item[gid] = ex

    # 1) Unificar PER por canonical name (primera mención manda)
    per2gid, next_gid = {}, 0
    for ex in cleaned:
        if ex.extraction_class == "PER":
            key = _norm_person(ex.extraction_text)
            if key not in per2gid:
                per2gid[key] = next_gid
                next_gid += 1
            ex.group_index = per2gid[key]
    # re-map todos los que refieren a PER previas:
    gid_map = {}
    for ex in cleaned:
        if ex.extraction_class == "PER":
            gid_map.setdefault(ex.group_index, per2gid[_norm_person(ex.extraction_text)])
    for ex in cleaned:
        if ex.group_index in gid_map:
            ex.group_index = gid_map[ex.group_index]

    # 2) Sticky por oración: atributo personal hereda gid de la última PER en la misma oración
    sent_spans = [(m.start(), m.end()) for m in re.finditer(r'[^.!?;\n]+[.!?;\n]?', full_text)]
    def in_span(e, span):
        ci = getattr(e, "char_interval", None)
        return ci and span[0] <= ci.start_pos < span[1]
    for s0, s1 in sent_spans:
        last_gid = None
        for ex in sorted([e for e in cleaned if in_span(e,(s0,s1))], key=lambda e: e.char_interval.start_pos if e.char_interval else 10**12):
            if ex.extraction_class == "PER":
                last_gid = ex.group_index
            elif ex.extraction_class in PERSON_ATTRS and last_gid is not None:
                ex.group_index = last_gid

    # 3) Filtrar clases ilegales y reindexar 0..K-1 por orden de aparición
    filtered = [e for e in cleaned if e.extraction_class in ALLOWED]
    seen, remap, nxt = {}, {}, 0
    for ex in filtered:
        g = ex.group_index
        if g is None:  # atributos sueltos podrían no tener gid; dejalos como están
            continue
        if g not in seen:
            seen[g] = nxt; nxt += 1
        ex.group_index = seen[g]
    return filtered


In [11]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]
docs_file

['document-06.docx',
 'document-07.docx',
 'aymurai - ejemplo 03.docx',
 'aymurai - ejemplo 02.docx',
 'document-02.docx',
 'document-03.docx',
 'document-04.docx',
 'document-08.docx']

In [12]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]

docs_file
documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars


In [13]:
import time

start = timeit.timeit()
predictions = {}
results = {}

name_doc = 'aymurai - ejemplo 02.docx'
text = joined_texts[name_doc]
print(text)

JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar
SENTENCIA
En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.
I. ANTECEDENTES
Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial.
La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó 

In [14]:
import time

start = timeit.timeit()
predictions = {}
results = {}

In [ ]:
#results['v5' + name_doc], predictions['v5'+name_doc] = resultv5, predictions

In [15]:
the_prompt= LONG_PROMPT
version = 'LONG_PROMPT-'

name_doc = 'aymurai - ejemplo 02.docx'
text = joined_texts[name_doc]

results[version+name_doc], predictions[version+name_doc] = langextract_prediction(text, the_prompt, examples, openai_api_key)

# for name_doc, text in joined_texts.items():
#     print(name_doc)
#     results[name_doc], predictions[name_doc] = langextract_prediction(text, PROMPT, examples, openai_api_key)
#     time.sleep(25)  # Sleep for 2 seconds to avoid rate limit

end = timeit.timeit()
print('\n Total time: ', end-start)


/var/folders/yf/2cttyjmx5j35yhq4st33mdmm0000gn/T/ipykernel_46647/2826528060.py:58: DeprecationWarning: 'language_model_type' is deprecated and will be removed in v2.0.0. Use model, config, or model_id parameters instead.
  result =  lx.extract(


[15:34:32] INFO     Starting sequential extraction passes for improved recall with 2 passes.      ]8;id=10997;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=818628;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#404\404]8;;\

           INFO     Starting extraction pass 1 of 2                                               ]8;id=100064;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=776417;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#416\416]8;;\

           INFO     Starting document annotation.                                                 ]8;id=400670;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=827786;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#261\261]8;;\

           INFO     Processing batch 0 with length 4                                              ]8;id=790659;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=869063;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#282\282]8;;\

...

[15:34:46] INFO     Completed alignment process for the provided source_text.                       ]8;id=784393;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/resolver.py\resolver.py]8;;\:]8;id=694604;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/resolver.py#305\305]8;;\

           INFO     Finalizing annotation for document ID doc_99624a2e.                           ]8;id=565214;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=308529;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#379\379]8;;\

           INFO     Document annotation completed.                                                ]8;id=504809;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=138514;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#390\390]8;;\

           INFO     Sequential extraction passes completed.                                       ]8;id=414566;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=693531;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#462\462]8;;\


 Total time:  -0.004910416027996689


In [16]:
pprint(results['LONG_PROMPT-aymurai - ejemplo 02.docx'])

AnnotatedDocument(
│   extractions=[
│   │   Extraction(
│   │   │   extraction_class='LOC',
│   │   │   extraction_text='CIUDAD DE SAN LORENZO',
│   │   │   char_interval=CharInterval(start_pos=31, end_pos=52),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=1,
│   │   │   group_index=0,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='NUM_EXPEDIENTE',
│   │   │   extraction_text='3187/2023',
│   │   │   char_interval=CharInterval(start_pos=68, end_pos=77),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=2,
│   │   │   group_index=1,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='LOC',
│   │   │   extraction_text='San Lorenzo',
│   │   │   char_interval=CharInterval(start_pos=188, end_pos=199),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=3,
│   │   │   group_index=2,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='LOC',
│   │   │   extraction_text='Provincia de Santa Fe',
│   │   │   char_interval=CharInterval(start_pos=201, end_pos=222),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=4,
│   │   │   group_index=3,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='FECHA',
│   │   │   extraction_text='22 días del mes de noviembre de 2023',
│   │   │   char_interval=CharInterval(start_pos=230, end_pos=266),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=5,
│   │   │   group_index=4,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Verónica Salvatierra',
│   │   │   char_interval=CharInterval(start_pos=322, end_pos=342),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=6,
│   │   │   group_index=5,
│   │   │   description=None,
│   │   │   attributes={'ROL_PROCESAL': 'Juez'}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='FECHA',
│   │   │   extraction_text='10 de noviembre de 2023',
│   │   │   char_interval=CharInterval(start_pos=522, end_pos=545),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=7,
│   │   │   group_index=6,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Ana Carolina Rodríguez',
│   │   │   char_interval=CharInterval(start_pos=555, end_pos=577),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=8,
│   │   │   group_index=7,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='DNI',
│   │   │   extraction_text='34.112.456',
│   │   │   char_interval=CharInterval(start_pos=583, end_pos=593),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=9,
│   │   │   group_index=8,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='DIRECCION',
│   │   │   extraction_text='calle Belgrano 785, Barrio Centro, San Lorenzo',
│   │   │   char_interval=CharInterval(start_pos=612, end_pos=658),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=10,
│   │   │   group_index=9,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Diego Esteban Fernández',
│   │   │   char_interv

In [97]:
results['LONG_PROMPT-aymurai - ejemplo 02.docx'].extractions

[Extraction(extraction_class='PER', extraction_text='Rodríguez, Ana Carolina', char_interval=CharInterval(start_pos=88, end_pos=111), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, attributes={}),
 Extraction(extraction_class='PER', extraction_text='Fernández, Diego Esteban', char_interval=CharInterval(start_pos=115, end_pos=139), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={}),
 Extraction(extraction_class='LOC', extraction_text='CIUDAD DE SAN LORENZO', char_interval=CharInterval(start_pos=178, end_pos=199), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=3, group_index=2, description=None, attributes={}),
 Extraction(extraction_class='NUM_EXPEDIENTE', extraction_text='3187/2023', char_interval=CharInterval(start_pos=68, end_pos=77), alignment_status=<AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>, extracti

In [100]:
results.keys()

dict_keys(['v6aymurai - ejemplo 02.docx', 'v5aymurai - ejemplo 02.docx', 'v7aymurai - ejemplo 02.docx', 'v8aymurai - ejemplo 02.docx', 'LONG_PROMPT-aymurai - ejemplo 02.docx'])

In [101]:
import pickle
with open("results_openai-multipleVersions-rionegro.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
extractions = [e for e in results['v8aymurai - ejemplo 02.docx'].extractions]
text = results['v8aymurai - ejemplo 02.docx'].text
clean_extractions = sanitize_and_fold(extractions,text)

In [98]:
pprint(results['LONG_PROMPT-aymurai - ejemplo 02.docx'])


AnnotatedDocument(
│   extractions=[
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Rodríguez, Ana Carolina',
│   │   │   char_interval=CharInterval(start_pos=88, end_pos=111),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=1,
│   │   │   group_index=0,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Fernández, Diego Esteban',
│   │   │   char_interval=CharInterval(start_pos=115, end_pos=139),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=2,
│   │   │   group_index=1,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='LOC',
│   │   │   extraction_text='CIUDAD DE SAN LORENZO',
│   │   │   char_interval=CharInterval(start_pos=178, end_pos=199),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=3,
│   │   │   group_index=2,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='NUM_EXPEDIENTE',
│   │   │   extraction_text='3187/2023',
│   │   │   char_interval=CharInterval(start_pos=68, end_pos=77),
│   │   │   alignment_status=<AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>,
│   │   │   extraction_index=4,
│   │   │   group_index=3,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='LOC',
│   │   │   extraction_text='San Lorenzo',
│   │   │   char_interval=CharInterval(start_pos=41, end_pos=52),
│   │   │   alignment_status=<AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>,
│   │   │   extraction_index=5,
│   │   │   group_index=4,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='LOC',
│   │   │   extraction_text='Provincia de Santa Fe',
│   │   │   char_interval=CharInterval(start_pos=201, end_pos=222),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=6,
│   │   │   group_index=5,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Verónica Salvatierra',
│   │   │   char_interval=CharInterval(start_pos=322, end_pos=342),
│   │   │   alignment_status=<AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>,
│   │   │   extraction_index=7,
│   │   │   group_index=6,
│   │   │   description=None,
│   │   │   attributes={'ROL_PROCESAL': 'Juez'}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='FECHA',
│   │   │   extraction_text='22 días del mes de noviembre de 2023',
│   │   │   char_interval=CharInterval(start_pos=230, end_pos=266),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=8,
│   │   │   group_index=7,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='FECHA',
│   │   │   extraction_text='10 de noviembre de 2023',
│   │   │   char_interval=CharInterval(start_pos=522, end_pos=545),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=9,
│   │   │   group_index=8,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Ana Carolina Rodríguez',
│   │   │   char_interval=CharInterval(start_pos=555, end_pos=577),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=10,
│   │   │   group_index=9,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='DNI',
│   │   │   extraction_text='34.112.456',
│   │   │   char_interval=CharInterval(start_pos=583

In [99]:
v = 'LONG_PROMPT-'#'v7'
result = results[v+'aymurai - ejemplo 02.docx']
extractions_name = f'examples-visualizations/extractions-{v}-rionegro-02.json'
extractions_html = f'examples-visualizations/extractions-{v}-rionegro-02.html'

print(f"Extracted {len(result.extractions)} entities from {len(result.text):,} characters")

# Save and visualize the results
lx.io.save_annotated_documents([result], output_name=extractions_name, output_dir=".")

# Generate the interactive visualization
html_content = lx.visualize(extractions_name)

with open(extractions_html, "w") as f:
    if hasattr(html_content, 'data'):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

print("Interactive visualization saved to test_extractions.html")

Extracted 33 entities from 3,593 characters


LangExtract: Saving to extractions-LONG_PROMPT--rionegro-02.json: 1 docs [00:00, 234.25 docs/s]

✓ Saved 1 documents to extractions-LONG_PROMPT--rionegro-02.json



LangExtract: Loading extractions-LONG_PROMPT--rionegro-02.json: 100%|██████████| 12.3k/12.3k [00:00<00:00, 11.9MB/s]

✓ Loaded 1 documents from extractions-LONG_PROMPT--rionegro-02.json
Interactive visualization saved to test_extractions.html


In [82]:
#resultv5 = result
[(f'extraction_text: {e.extraction_text}, group_index: {e.group_index}, attributes: {e.attributes}') for e in result.extractions if e.extraction_class == 'PER']

['extraction_text: Rodríguez, Ana Carolina, group_index: 0, attributes: None',
 'extraction_text: Fernández, Diego Esteban, group_index: 1, attributes: None',
 'extraction_text: Verónica Salvatierra, group_index: 7, attributes: None',
 'extraction_text: Ana Carolina Rodríguez, group_index: 9, attributes: None',
 'extraction_text: Diego Esteban Fernández, group_index: 12, attributes: None',
 'extraction_text: M.F.R., group_index: 15, attributes: None',
 "extraction_text: Patricia Gómez, group_index: 7, attributes: {'ROL_PROCESAL': 'Testigo'}",
 'extraction_text: Luis Alberto Rivas, group_index: 0, attributes: {}',
 "extraction_text: Diego Esteban Fernández, group_index: 1, attributes: {'ROL_PROCESAL': 'Imputado'}",
 "extraction_text: Ana Carolina Rodríguez, group_index: 2, attributes: {'ROL_PROCESAL': 'Victima'}",
 'extraction_text: Diego Esteban Fernández, group_index: 0, attributes: {}',
 'extraction_text: Ana Carolina Rodríguez, group_index: 1, attributes: {}',
 "extraction_text: M.F

In [68]:
[e for e in resultv5.extractions if e.extraction_class == 'DNI']

[Extraction(extraction_class='DNI', extraction_text='34.112.456', char_interval=CharInterval(start_pos=583, end_pos=593), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=14, group_index=8, description=None, attributes=None),
 Extraction(extraction_class='DNI', extraction_text='31.998.210', char_interval=CharInterval(start_pos=719, end_pos=729), alignment_status=<AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>, extraction_index=20, group_index=11, description=None, attributes=None)]

In [70]:
import pickle
with open("results_openai-v5-rionegro02.pkl", "wb") as f:
    pickle.dump(resultv5, f)

In [ ]:
import pickle

result_path = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/notebooks/experiments/anonymization/results_openai-attributes-v5-rionegro02.pkl'

with open(result_path,"rb") as file:
    results = pickle.load(file)

In [ ]:
pprint(results['documento-entrerios-02.docx'])